In [ ]:
#Basic from scratch decision tree classifier implementation in Python


#Dependencies
import math
import numpy as np
import pandas as pd


In [ ]:
#First we define the node class

class Node():

    def __init__(self,threshhold: float = None, Leaf: bool = False, ds_y: list = [], index:int = 0, ds_X: list= []):

        self.threshold = threshhold
        self.Leaf = Leaf
        self.ds_y = ds_y
        self.ds_X = ds_X
        self.index = index
        self.left = None
        self.right = None

    def evaluate(self, value):
        feature_val = value[self.index]

        if self.Leaf == True:
            self.leaf_node()
            return self.prediction
        
        if feature_val < self.threshold:
            return self.left.evaluate(value)
        else:
            return self.right.evaluate(value)
        
    def leaf_node(self):
        
        ds_y = self.ds_y
        class_dict: dict = {}
        total_instances: int = len(ds_y)


        rows, values = np.unique(ds_y, return_counts=True)

        for classification, count in zip(rows,values):
            class_dict[classification] = count/total_instances
        top_prediction = list(class_dict.values())
        top_class = list(class_dict.keys())
        self.prediction = top_class[top_prediction.index(max(top_prediction))]
        self.class_dict = class_dict
        

class DecisionTree():

    def __init__(self):
        pass

    def __str__(self):
        return 'You have not trained a model yet'

    def gini(self, ds_y: list) -> float:
        '''
        Gini = 1 - Sum((proportion of set)**2)
        '''
        gini:float = 0.0
        no_elements: float = len(ds_y)
        rows, values = np.unique(ds_y, return_counts=True)


        for index, row in enumerate(rows):
            gini += (values[index]/no_elements)**2

        return 1-gini

    def weighted_gini(self, ds_y_L, ds_y_R):

        gini_L = self.gini(ds_y_L)
        gini_R = self.gini(ds_y_R)

        proportion_left = len(ds_y_L)
        proportion_right = len(ds_y_R)
        total_proportion = proportion_left+proportion_right
        
        weighted_gini =(((gini_L)*(proportion_left/total_proportion))+((gini_R)*((proportion_right/total_proportion))))
        
        return weighted_gini

    def split(self, ds_X, ds_y):
        '''
        This 

        For every value calculate gini, you need the array index value
        '''
        best_gini = 1000000
        return_list = None


        for feature_index in range(len(ds_X[0])):
            threshholds = np.unique(ds_X[:, feature_index])
            for threshhold in threshholds:
                mask_L = ds_X[:, feature_index] <= threshhold
                mask_R = ~mask_L

                if np.any(mask_L) and np.any(mask_R):
                    weighted_gini = self.weighted_gini(ds_y[mask_L],ds_y[mask_R])
                    if weighted_gini<best_gini:
                        best_gini = weighted_gini
                        return_list = [best_gini,threshhold, feature_index, ds_X[mask_L],ds_X[mask_R],ds_y[mask_L],ds_y[mask_R]]


        return None if return_list == None else return_list

    def stopping_condition(self, max_depth, min_samples_split, min_samples_leaf, min_impurity_split):
        
        pass

    def fit(self, ds_X = None, ds_y = None, model = None, max_depth = 5, min_samples_split = 3, min_samples_leaf = 3, min_impurity_split = 0.1, previous_node = []):
        #[weighted_sum, threshhold, threshhold_index, ds_X_L, ds_X_R, ds_Y_L, ds_Y_R]

        if model == None:
            try:
                ds_X = ds_X.to_numpy()
                ds_y = ds_y.to_numpy()
                model = Node(ds_y = ds_y , ds_X = ds_X)
                self.model = model
            except:
                model = Node(ds_y = ds_y , ds_X = ds_X)
                self.model = model
        ''' 
        Could turn stopping condition into a function for modularity
        '''
       # if len(ds_y) > min_samples_leaf and isinstance(previous_node, list) is False:
        #    previous_node.leaf_node()
         #   return
        
        
        if len(model.ds_y) < min_samples_split:
            model.Leaf = True
            return 

        if self.gini(model.ds_y) < min_impurity_split:
            model.Leaf = True
            return 

        if max_depth > 0: 
            split = self.split(model.ds_X, model.ds_y)

            if split == None:
                model.Leaf = True
                return

            model.threshold = split[1]

            model.index = split[2]
            if max_depth == 1:
            
                model.left = Node(ds_y = split[5], ds_X = split[3], Leaf = True)

                model.right = Node(ds_y = split[6], ds_X = split[4], Leaf = True)

                model.left.leaf_node()
                model.right.leaf_node()
                return
            else:
                model.left = Node(ds_y = split[5], ds_X = split[3])

                model.right = Node(ds_y = split[6], ds_X = split[4])
            
            self.fit(model = model.left, max_depth=max_depth-1, min_impurity_split = min_impurity_split, min_samples_split = min_samples_split, previous_node = model, min_samples_leaf = min_samples_leaf, ds_y = model.left.ds_y, ds_X = model.left.ds_X)
            #L and R
            self.fit(model = model.right, max_depth=max_depth-1, min_impurity_split = min_impurity_split, min_samples_split = min_samples_split, previous_node = model, min_samples_leaf = min_samples_leaf, ds_y = model.right.ds_y, ds_X = model.right.ds_X)

    def predict(self, df_X):
        predictions: list = []
        for predicted_X in df_X:
            predictions.append(self.model.evaluate(predicted_X))
        
        return predictions
    
    def train_test_split():
        pass


    def accuracy_score(df_y, predictions):
        pass

